# Approach 2a1 — Average per-run fitted curves, then refit

**Pipeline:** Each run already fit individually in `fitting_function_IPA.ipynb` → `<CE>` at each BN is `Mean_Fit_CE` in `mean_fit_p_{p}_bs_{bs}.csv` → refit that averaged curve to a single `A, B, n` → IPA.

**Input:** `Fitting_IPA_curves_data_I/BS_{bs}/mean_fit_p_{p}_bs_{bs}.csv`.

**Difference from 2a2:** here we refit the averaged-of-fits curve; 2a2 just averages parameters directly.

In [1]:
# === Cell 1 — Config, imports, helpers ===
import os, glob, re
import numpy as np
import pandas as pd
from lmfit import Parameters, minimize
import warnings
warnings.filterwarnings("ignore")

# Paths
BASE_DIR = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\prune_layers_ALL"
FIT_DIR  = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\Fitting_IPA_curves_data_I"
OUT_DIR  = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_2a1"
INTERMEDIATE_DIR = os.path.join(OUT_DIR, "intermediate")
os.makedirs(INTERMEDIATE_DIR, exist_ok=True)

BATCH_SIZES = [64, 1024, 60000]
CE_o = np.log(10)   # max CE for 10-class problem, ~2.302585

# Auto-detect pruning percentages from prune_layers_ALL/p-percentage_*/
p_dirs = glob.glob(os.path.join(BASE_DIR, "p-percentage_*"))
PRUNING_LEVELS = sorted([
    float(re.search(r"p-percentage_([\d.]+)", d).group(1))
    for d in p_dirs
])
print(f"Found {len(PRUNING_LEVELS)} pruning percentages: {PRUNING_LEVELS}")
print(f"CE_o = ln(10) = {CE_o:.6f}")

# Fit-function helpers (verbatim from fitting_function_IPA.ipynb)
A_MIN, A_MAX = 0.1, 2.3
B_MIN, B_MAX = 0, 1000
N_MIN, N_MAX = 0.5, 3.0


def initialize_guesses(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]
    A0 = np.percentile(y, 5)
    B0 = np.percentile(y, 95) - A0
    n0 = 0.5
    if len(x) > 10:
        denom = y[0] - A0
        if abs(denom) > 1e-10:
            frac = max(1e-6, (y[0] - y[-1]) / denom)
            if frac > 0:
                n0 = max(0.3, min(1.5, -np.log(frac)))
    return A0, n0, B0


def model(params, x):
    vals = params.valuesdict()
    A, B, n = vals["A"], vals["B"], vals["n"]
    return A + B / ((x + 1) ** n)


def residual(params, x, data):
    weight = x
    return weight * (model(params, x) - data)


def fit_curve(x, y):
    mask  = ~np.isnan(y)
    x_fit = np.asarray(x)[mask]
    y_fit = np.asarray(y)[mask]
    if len(x_fit) < 10:
        return None
    A0, n0, B0 = initialize_guesses(x_fit, y_fit)
    params = Parameters()
    params.add("A", value=A0, min=A_MIN, max=A_MAX)
    params.add("B", value=B0, min=B_MIN, max=B_MAX)
    params.add("n", value=n0, min=N_MIN, max=N_MAX)
    try:
        return minimize(residual, params, args=(x_fit, y_fit))
    except Exception:
        return None

# --- IPA from fit (single source of truth) ---
# CE_L is the physically meaningful learning threshold.
#   CE_L = CE_o - 0.9 * (CE_o - A)
#   IPA  = abs(CE_o - CE_L) / learn_BN   where learn_BN = first BN in x_grid with fitted CE <= CE_L.
# We intentionally do NOT simplify to 0.9*(CE_o - A)/learn_BN — CE_L stays a first-class variable.
def compute_ipa_from_fit(x_grid, A, B, n):
    x_grid = np.asarray(x_grid, dtype=float)
    CE_L = CE_o - 0.9 * (CE_o - A)
    fitted = A + B / ((x_grid + 1) ** n)
    mask = fitted <= CE_L

    if mask.any():
        # In-range: original discrete-search behavior
        learn_BN = float(x_grid[mask][0])
        fitted_at = float(fitted[mask][0])
    else:
        # Out-of-range: extrapolate analytically, then ceil to integer BN
        denom = CE_L - A
        if denom <= 0 or n <= 0 or B <= 0:
            return {"CE_L": CE_L, "learn_BN": np.nan, "IPA": np.nan, "fitted_at_learn_BN": np.nan}
        BN_analytic = (B / denom) ** (1.0 / n) - 1.0
        if not np.isfinite(BN_analytic) or BN_analytic <= 0:
            return {"CE_L": CE_L, "learn_BN": np.nan, "IPA": np.nan, "fitted_at_learn_BN": np.nan}
        learn_BN = float(np.ceil(BN_analytic))
        fitted_at = float(A + B / ((learn_BN + 1) ** n))

    if learn_BN == 0:
        return {"CE_L": CE_L, "learn_BN": 0.0, "IPA": np.nan, "fitted_at_learn_BN": fitted_at}
    IPA = abs(CE_o - CE_L) / learn_BN
    return {"CE_L": CE_L, "learn_BN": learn_BN, "IPA": IPA, "fitted_at_learn_BN": fitted_at}
print("Cell 1 ready.")


Found 19 pruning percentages: [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.82, 0.84, 0.86, 0.88, 0.9, 0.92, 0.94, 0.96, 0.98, 1.0]
CE_o = ln(10) = 2.302585
Cell 1 ready.


In [2]:
# === Cell 2 — Approach 2a1: refit the per-run-mean fitted curve, then IPA ===
# Per (P%, BS): load mean_fit_p_{p}_bs_{bs}.csv (already <CE> across 100 fits at each BN),
# refit Mean_Fit_CE vs Batch_Number with fit_curve, then IPA from the refitted curve.
inter_by_bs = {}

for bs in BATCH_SIZES:
    print("\n" + "=" * 70)
    print(f"  Approach 2a1 — Batch size {bs}")
    print("=" * 70)
    rows = []
    for p in PRUNING_LEVELS:
        mean_csv = os.path.join(FIT_DIR, f"BS_{bs}", f"mean_fit_p_{p}_bs_{bs}.csv")
        if not os.path.exists(mean_csv):
            print(f"  [SKIP] P%={p*100:5.1f}%  — missing {mean_csv}")
            continue
        df = pd.read_csv(mean_csv)
        df.columns = df.columns.str.strip()
        df = df.dropna(subset=["Batch_Number", "Mean_Fit_CE"])
        x = df["Batch_Number"].values.astype(float)
        y = df["Mean_Fit_CE"].values.astype(float)

        result = fit_curve(x, y)
        if result is None:
            print(f"  [FAIL] P%={p*100:5.1f}%  — refit did not converge")
            continue
        A = result.params["A"].value
        B = result.params["B"].value
        n = result.params["n"].value
        ipa = compute_ipa_from_fit(x, A, B, n)

        print(f"  P%={p*100:5.1f}%  A={A:.4f}  B={B:.4f}  n={n:.4f}  "
              f"CE_o={CE_o:.4f}  CE_L={ipa['CE_L']:.4f}  learn_BN={ipa['learn_BN']!r:>8}  IPA={ipa['IPA']}")

        rows.append({
            "P%": p * 100, "A": A, "B": B, "n": n,
            "CE_o": CE_o, "CE_L": ipa["CE_L"],
            "learn_BN": ipa["learn_BN"], "fitted_at_learn_BN": ipa["fitted_at_learn_BN"],
            "IPA": ipa["IPA"],
        })

    if rows:
        bs_df = pd.DataFrame(rows)
        inter_path = os.path.join(INTERMEDIATE_DIR, f"approach_2a1_refit_params_bs_{bs}.csv")
        bs_df.to_csv(inter_path, index=False)
        print(f"  Saved: {inter_path}")
        inter_by_bs[bs] = bs_df

print("\n[Cell 2 done]")



  Approach 2a1 — Batch size 64
  P%=  0.0%  A=0.2972  B=5.3657  n=0.8827  CE_o=2.3026  CE_L=0.4977  learn_BN=    41.0  IPA=0.04402046182348596
  P%= 10.0%  A=0.2978  B=5.3167  n=0.8724  CE_o=2.3026  CE_L=0.4983  learn_BN=    42.0  IPA=0.042959662739405165
  P%= 20.0%  A=0.2851  B=5.2918  n=0.8307  CE_o=2.3026  CE_L=0.4869  learn_BN=    51.0  IPA=0.035602157372184744
  P%= 30.0%  A=0.2874  B=5.3590  n=0.8204  CE_o=2.3026  CE_L=0.4889  learn_BN=    54.0  IPA=0.03358687018694848
  P%= 40.0%  A=0.2837  B=5.6842  n=0.8033  CE_o=2.3026  CE_L=0.4856  learn_BN=    63.0  IPA=0.028841530856494657
  P%= 50.0%  A=0.2787  B=6.3285  n=0.7840  CE_o=2.3026  CE_L=0.4811  learn_BN=    80.0  IPA=0.022768371041835607
  P%= 60.0%  A=0.2729  B=6.9022  n=0.7480  CE_o=2.3026  CE_L=0.4758  learn_BN=   111.0  IPA=0.01645710555003645
  P%= 70.0%  A=0.2849  B=7.7383  n=0.7279  CE_o=2.3026  CE_L=0.4866  learn_BN=   149.0  IPA=0.012187592169569248
  P%= 80.0%  A=0.3100  B=8.1977  n=0.6711  CE_o=2.3026  CE_L=0.5093

In [3]:
# === Cell 3 — Build wide summary CSV for Approach 2a1 ===
# Schema: P%, IPA_Avg_64, STD_64, IPA_Avg_1024, STD_1024, IPA_Avg_60000, STD_60000
# STD columns blank (NaN) for single-curve approaches; populated only by 2b.
summary_rows = []
for p in PRUNING_LEVELS:
    row = {"P%": p * 100}
    for bs in BATCH_SIZES:
        df = inter_by_bs.get(bs)
        if df is None:
            mean_val, std_val = np.nan, np.nan
        else:
            sub = df[df["P%"] == p * 100]
            mean_val = float(sub["IPA"].iloc[0]) if (not sub.empty and "IPA" in sub.columns) else (
                       float(sub["IPA_mean"].iloc[0]) if (not sub.empty and "IPA_mean" in sub.columns) else np.nan)
        std_val  = np.nan
        row[f"IPA_Avg_{bs}"] = mean_val
        row[f"STD_{bs}"]     = std_val
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows, columns=[
    "P%", "IPA_Avg_64", "STD_64", "IPA_Avg_1024", "STD_1024", "IPA_Avg_60000", "STD_60000"
])
out_csv = os.path.join(OUT_DIR, "ipa_summary_approach_2a1.csv")
summary_df.to_csv(out_csv, index=False)
print(f"\nFinal summary written: {out_csv}")
print(summary_df.to_string(index=False))



Final summary written: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_2a1\ipa_summary_approach_2a1.csv
   P%  IPA_Avg_64  STD_64  IPA_Avg_1024  STD_1024  IPA_Avg_60000  STD_60000
  0.0    0.044020     NaN      0.063223       NaN       0.072309        NaN
 10.0    0.042960     NaN      0.059257       NaN       0.064896        NaN
 20.0    0.035602     NaN      0.054152       NaN       0.057058        NaN
 30.0    0.033587     NaN      0.047323       NaN       0.050899        NaN
 40.0    0.028842     NaN      0.044875       NaN       0.043748        NaN
 50.0    0.022768     NaN      0.128363       NaN       0.037578        NaN
 60.0    0.016457     NaN      0.032635       NaN       0.030188        NaN
 70.0    0.012188     NaN      0.024181       NaN       0.026417        NaN
 80.0    0.007060     NaN      0.013192       NaN       0.018143        NaN
 82.0    0.006028     NaN      0.011230       NaN       0.014345        NaN
 84.0    0.005068    

In [4]:
# === Cell 4 — Plot IPA vs P% ===
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

TAG   = "2a1"
TITLE = "Approach 2a1 — Refit averaged per-run fitted curves"
BS_COLOR = {64: "#1f77b4", 1024: "#d62728", 60000: "#2ca02c"}

plt.rcParams.update({"font.size": 14})
fig, ax = plt.subplots(figsize=(10, 6))

for bs in BATCH_SIZES:
    mean_col = f"IPA_Avg_{bs}"
    sub = summary_df.dropna(subset=[mean_col])
    if sub.empty:
        continue
    ax.plot(sub["P%"].values, sub[mean_col].values,
            label=f"BS={bs}", color=BS_COLOR[bs], marker="o", markersize=6, linewidth=2)

ax.set_xlabel("Pruning Percentage (%)")
ax.set_ylabel("IPA")
# ax.set_yscale("log")
ax.set_title(TITLE, fontsize=14)
ax.grid(True, which="both", alpha=0.3)
ax.legend(frameon=False)

out_png = os.path.join(OUT_DIR, f"ipa_plot_approach_{TAG}.png")
plt.tight_layout()
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out_png}")

Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_2a1\ipa_plot_approach_2a1.png
